# 📊 Delivery Delay Prediction — Business Dashboard

This notebook visualizes insights from the **Gold aggregation tables** built after ML inference.

**5 Dashboard Focus Areas (driven by actual data patterns):**
1. **Geographic Risk** — AL/MA/SE states have 3x national average late rate
2. **Model Performance** — Predicted vs actual late deliveries from ML output
3. **Category Risk** — `audio`, `home_comfort`, `baby` are highest-risk categories
4. **Seasonal Patterns** — Mar (14.6%) and Feb/Nov (~12%) are peak delay months
5. **High-Risk Order Profiles** — Premium price + tight delivery window = worst combination

> All `display()` calls render as **interactive Databricks charts**. Switch each cell from Table → Chart using the chart icon, then configure axes as described.

---

In [0]:
from pyspark.sql import functions as F

CATALOG = "project"
GOLD    = "gold"

def gold(t): return spark.read.table(f"{CATALOG}.{GOLD}.{t}")

print("Ready.")

Ready.


---
# Focus 1 — Geographic Delivery Risk

The data shows a stark North-South divide in Brazil. Northern states (AL, MA, SE, CE) face late rates above 15%, while southern states (SP, SC, PR) stay below 8%. This is the single strongest predictor in the dataset.

**What this tells the business:** Logistics capacity in northern Brazil needs to be reinforced, or estimated delivery windows widened to reduce customer disappointment.

In [0]:
# Chart: Bar chart → X: customer_state, Y: late_rate_pct | Color: risk_tier
# Sort by late_rate_pct descending
display(
    gold("agg_late_by_state")
    .select("customer_state", "total_orders", "late_orders",
            "late_rate_pct", "avg_freight", "avg_estimated_days", "risk_tier")
    .orderBy(F.col("late_rate_pct").desc())
)

customer_state,total_orders,late_orders,late_rate_pct,avg_freight,avg_estimated_days,risk_tier
AL,414,86,20.77,35.93,33.1,High
MA,778,141,18.12,38.6,31.5,High
SE,363,61,16.8,36.84,31.4,High
CE,1398,193,13.81,32.73,32.0,Medium
PI,511,70,13.7,39.35,30.9,Medium
BA,3559,428,12.03,26.4,30.2,Medium
RJ,13599,1585,11.66,20.95,27.1,Medium
RR,44,5,11.36,42.28,46.9,Medium
PA,1029,115,11.18,35.77,38.0,Medium
PB,572,63,11.01,43.19,33.6,Medium


Databricks visualization. Run in Databricks to view.

In [0]:
# Supplementary: Revenue vs Late Rate — which high-revenue states are also high-risk?
# Chart: Scatter plot → X: total_revenue, Y: late_rate_pct, Details: customer_state
display(
    gold("agg_late_by_state")
    .select("customer_state", "total_revenue", "late_rate_pct", "total_orders", "risk_tier")
)

customer_state,total_revenue,late_rate_pct,total_orders,risk_tier
AL,76002.17,20.77,414,High
MA,114655.22,18.12,778,High
SE,55545.19,16.8,363,High
CE,216848.68,13.81,1398,Medium
PI,83623.79,13.7,511,Medium
BA,479888.08,12.03,3559,Medium
RJ,1699214.93,11.66,13599,Medium
RR,6721.47,11.36,44,Medium
PA,172413.4,11.18,1029,Medium
PB,110134.4,11.01,572,Medium


Databricks visualization. Run in Databricks to view.

---
# Focus 2 — Model Performance: Predicted vs Actual

This is the key evaluation of the ML system — does the model correctly identify which orders will arrive late? We compare `predicted_is_late` vs `actual_is_late` from the inference output table, broken down by state so we can see where the model is most and least confident.

**What this tells the business:** High-risk states with strong model accuracy can be acted upon immediately — proactive alerts to customers before delays happen.

In [0]:
# Overall model KPI card
# Chart: Single-value / table card
display(gold("agg_model_overall_performance"))

total_predictions,total_actual_late,total_predicted_late_top10pct,avg_late_probability_pct,accuracy_top10pct_precision
21429,1459,2331,44.09,87.19


In [0]:
# Predicted vs Actual late rate by state
# Chart: Grouped Bar → X: customer_state, Y1: actual_late_rate_pct, Y2: predicted_late_rate_pct
display(
    gold("agg_model_performance_by_state")
    .select("customer_state", "actual_late_rate_pct",
            "predicted_late_top10pct", "avg_late_probability_pct", "accuracy_pct")
    .orderBy(F.col("actual_late_rate_pct").desc())
)

customer_state,actual_late_rate_pct,predicted_late_top10pct,avg_late_probability_pct,accuracy_pct
AL,20.88,17,51.15,78.02
MA,20.39,52,52.75,71.71
CE,15.31,80,51.42,78.44
ES,12.65,63,48.45,79.63
PB,12.5,16,48.52,76.79
PI,12.38,11,46.28,82.86
BA,11.68,157,53.0,75.64
RN,11.43,12,44.69,79.05
RJ,11.4,915,53.03,71.67
RR,11.11,0,37.84,88.89


Databricks visualization. Run in Databricks to view.

In [0]:
# Average predicted late probability distribution by state
# Chart: Bar chart → X: customer_state, Y: avg_late_probability_pct
display(
    gold("agg_model_performance_by_state")
    .select("customer_state", "avg_late_probability_pct", "total_orders")
    .orderBy(F.col("avg_late_probability_pct").desc())
)

customer_state,avg_late_probability_pct,total_orders
RJ,53.03,2736
BA,53.0,702
MA,52.75,152
CE,51.42,320
AL,51.15,91
MS,50.42,176
SC,49.19,851
PB,48.52,112
ES,48.45,427
TO,48.28,80


Databricks visualization. Run in Databricks to view.

---
# Focus 3 — Product Category Risk

`audio` has the highest late rate (11.9%), followed by `home_comfort` (9.7%) and `baby` (7.8%). Critically, `health_beauty` is both high-volume (9,183 orders) and above-average risk (7.6%) — making it the **highest absolute number of delayed deliveries** in any single category.

**What this tells the business:** Category managers should negotiate SLA improvements with suppliers/carriers for flagged categories, especially `baby` (safety-critical) and `health_beauty` (high volume).

In [0]:
# Top 15 categories by late rate
# Chart: Horizontal Bar → X: late_rate_pct, Y: product_category_name_english | Color: revenue_risk_flag
display(
    gold("agg_late_by_category")
    .select("product_category_name_english", "total_orders", "late_rate_pct",
            "total_revenue", "avg_freight", "freight_to_price_pct", "revenue_risk_flag")
    .orderBy(F.col("late_rate_pct").desc())
    .limit(15)
)

product_category_name_english,total_orders,late_rate_pct,total_revenue,avg_freight,freight_to_price_pct,revenue_risk_flag
audio,354,11.86,50123.2,15.75,11.12,High Risk
fashion_underwear_beach,118,10.17,9036.85,14.58,19.04,Normal
christmas_supplies,148,10.14,8644.17,21.22,36.32,Normal
home_confort,413,9.69,55606.35,19.64,14.59,Normal
office_furniture,1630,7.85,261743.79,40.11,24.98,Normal
baby,2901,7.79,391835.28,22.32,16.52,High Revenue
electronics,2685,7.64,152781.74,16.78,29.49,Normal
health_beauty,9183,7.6,1202974.41,18.97,14.48,High Revenue
unknown_category,1502,7.59,170708.72,17.63,15.51,Normal
construction_tools_lights,294,7.48,38631.12,24.65,18.76,Normal


Databricks visualization. Run in Databricks to view.

In [0]:
# Revenue vs Late Rate scatter — find categories that are BOTH high value and high risk
# Chart: Scatter → X: total_revenue, Y: late_rate_pct, Details: product_category_name_english
display(
    gold("agg_late_by_category")
    .select("product_category_name_english", "total_revenue",
            "late_rate_pct", "total_orders", "revenue_risk_flag")
)

product_category_name_english,total_revenue,late_rate_pct,total_orders,revenue_risk_flag
audio,50123.2,11.86,354,High Risk
fashion_underwear_beach,9036.85,10.17,118,Normal
christmas_supplies,8644.17,10.14,148,Normal
home_confort,55606.35,9.69,413,Normal
office_furniture,261743.79,7.85,1630,Normal
baby,391835.28,7.79,2901,High Revenue
electronics,152781.74,7.64,2685,Normal
health_beauty,1202974.41,7.6,9183,High Revenue
unknown_category,170708.72,7.59,1502,Normal
construction_tools_lights,38631.12,7.48,294,Normal


Databricks visualization. Run in Databricks to view.

---
# Focus 4 — Seasonal Late Rate Patterns

March (14.6%), February (11.9%), and November (12%) are the three worst months — likely driven by post-holiday demand spillover (Feb/Mar) and Black Friday buildup (Nov). June is the safest month at only 1.8% late rate.

**What this tells the business:** Pre-position inventory and expand carrier capacity from January through March and in November. Customer communication should be proactively adjusted in these months.

In [0]:
# Monthly trend — late rate + order volume
# Chart: Line chart → X: month_name, Y1: late_rate_pct, Y2: total_orders
# Highlights peak delay months
display(
    gold("genuine_problematic_months")
    .select("order_month", "month_name", "total_orders",
            "late_orders", "late_rate_pct", "total_revenue", "is_peak_delay_month")
    .orderBy("order_month")
)

order_month,month_name,total_orders,late_orders,late_rate_pct,total_revenue,is_peak_delay_month
1,Jan,8613,469,5.45,999501.49,0
2,Feb,9004,1069,11.87,1027512.42,0
3,Mar,10563,1546,14.64,1278454.93,0
4,Apr,10064,466,4.63,1272989.09,0
5,May,11388,600,5.27,1421044.8,0
6,Jun,10136,183,1.81,1242859.16,0
7,Jul,10964,351,3.2,1309082.1,0
8,Aug,11535,550,4.77,1351820.54,0
9,Sep,4515,204,4.52,582917.61,0
10,Oct,5292,202,3.82,666854.21,0


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

---
# Focus 5 — High-Risk Order Profiles

The riskiest combination in the data: **Premium price (200+ BRL) + Tight delivery window (≤10 days)**. Orders with `estimated_days ≤ 10` have a 10.5% late rate — the highest of any delivery window — because there's zero buffer for any logistics disruption. Higher-priced orders also face more scrutiny and stricter customer expectations.

**What this tells the business:** A rule-based early warning system should flag orders where `price > 200 AND estimated_days <= 10`. The model's `prob_late` score can be used as the trigger threshold (e.g. send alert if `prob_late > 0.4`).

In [0]:
# Risk matrix: price tier × delivery window
# Chart: Heatmap or Grouped Bar → X: price_tier, Group: delivery_window, Y: late_rate_pct
display(
    gold("price_delivery_risk_matrix")
    .select("price_tier", "delivery_window", "total_orders", "late_rate_pct", "avg_price")
    .orderBy(F.col("late_rate_pct").desc())
)

price_tier,delivery_window,total_orders,late_rate_pct,avg_price
Premium,Normal,6831,8.89,291.93
Mid,Normal,7342,8.34,56.19
Premium,Extended,6269,8.28,302.42
Premium,Tight,5798,7.83,294.4
High,Normal,7323,7.58,101.55
Mid,Extended,6086,7.31,56.49
Low,Normal,7075,7.22,26.04
High,Extended,6276,6.98,101.94
High,Tight,6446,6.72,100.96
Premium,Long,7584,6.04,309.17


Databricks visualization. Run in Databricks to view.

In [0]:
df = spark.read.table(f"{CATALOG}.{GOLD}.predictions")
from pyspark.ml.functions import vector_to_array

df = df.withColumn(
    "late_probability",
    vector_to_array(F.col("probability"))[1].cast("double")
)


In [0]:
# From predictions: orders the model flagged as HIGH probability of being late
# These are the actionable candidates for proactive intervention
# Chart: Table / Bar → X: customer_state, Y: count of high-risk orders
high_risk_pred = (
    df
    .filter(F.col("late_probability") >= 0.0665)   # high-confidence late predictions
    .groupBy("customer_state", "product_category_name_english")
    .agg(
        F.count("*").alias("high_risk_orders"),
        F.round(F.avg("late_probability") * 100, 2).alias("avg_late_prob_pct"),
        F.round(F.avg("price"), 2).alias("avg_order_value")
    )
    .orderBy(F.col("high_risk_orders").desc())
    .limit(20)
)
display(high_risk_pred)

customer_state,product_category_name_english,high_risk_orders,avg_late_prob_pct,avg_order_value
SP,bed_bath_table,966,39.03,87.95
SP,health_beauty,753,42.61,111.84
SP,housewares,647,36.58,86.46
SP,sports_leisure,644,40.37,108.8
SP,furniture_decor,635,40.72,80.47
SP,computers_accessories,624,40.2,104.64
SP,watches_gifts,413,40.82,182.68
SP,toys,350,39.12,107.0
SP,auto,330,42.23,121.35
SP,telephony,326,36.99,54.74


Databricks visualization. Run in Databricks to view.

---
# Dashboard Summary: Delivery Analytics

| # | Focus | Key Finding | Business Action |
|---|---|---|---|
| 1 | Geographic Risk |RJ (53.75% avg prob), MG (42.18% avg prob) vs SP (~39–42% range); regional variance significant | Strengthen carrier partnerships in high-risk states; tier SLA expectations |
| 2 | Model Probability | Top 10% predicted orders capture 17–915 flagged per state; varies widely | Use probability ranking; adjust alert thresholds by regional volume |
| 3 | Category Risk | audio: 11.86% late rate (354 orders, $50k revenue) — only "High Risk" category | Review audio supplier/category SLAs; flag for escalation |
| 4 | Seasonal Peaks | Q1 spike (Jan–Mar); Feb at peak risk; low risk Jun (~1.8%) | Scale capacity Jan–Mar; validate Jun low-volume performance |
| 5 | Order Profile Risk | Premium (>$200) + tight delivery (≤10d) = 8.28–8.89% late | Flag orders: price>200 AND estimated_days≤10 |

## Key Insights

- **High-Risk States**: AL, MA, SE have 4–5x worse performance than SP
- **Model Usage**: Probability scores outperform binary predictions for flagging interventions
- **Category Spotlight**: audio is sole "High Risk" category; all others normal
- **Seasonal Pattern**: Q1 (Jan–Mar) and Feb spike; Jun is lowest risk
- **Operational Alert**: Premium price + tight windows = operational bottleneck